# end-grad-default-ones-like — worked example 2: resolve_end_grad: Two Paths — Default vs. Explicit

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `end-grad-default-ones-like`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The entry point of a backpropagation engine needs to handle two situations: no `end_grad` supplied (default to `ones_like`) and an explicit `end_grad` supplied (unbox and validate). The shape check is necessary because an explicit seed of the wrong shape would silently corrupt every gradient downstream. Both paths return a raw array, not a wrapped tensor, so the rest of the engine works with consistent types.

## Worked solution

We implement and test the `resolve_end_grad` function.

**Path 1 — None:** When `end_grad is None`, we return `t.ones_like(end_node.array)`. The `.array` attribute holds the raw underlying tensor. The result is a fresh all-ones tensor with the same shape and dtype.

**Path 2 — explicit MiniTensor:** When `end_grad` is provided, we first assert the shapes match. A shape mismatch raises `AssertionError` with a helpful message listing both shapes. If shapes match, we return `end_grad.array` (unboxed).

**Why the shape assertion?** The explicit end_grad is consumed in exactly the same way as the ones seed — it multiplies through the chain rule. If it has the wrong shape, broadcasting might silently produce the wrong dimensions for every downstream gradient.

In [ ]:
import torch as t

# Minimal MiniTensor stand-in for illustration
class MiniTensor:
    def __init__(self, array):
        self.array = array

def resolve_end_grad(end_node: MiniTensor, end_grad):
    """Return the backward seed as a raw tensor."""
    if end_grad is None:
        return t.ones_like(end_node.array)
    assert end_grad.array.shape == end_node.array.shape, (
        f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
        f'end_node shape {tuple(end_node.array.shape)}'
    )
    return end_grad.array

# Test path 1: default
node = MiniTensor(t.tensor([1.0, 2.0, 3.0]))
seed = resolve_end_grad(node, None)
print(f"Default seed: {seed}")
assert seed.shape == (3,)
assert (seed == 1.0).all()

# Test path 2: explicit matching shape
explicit = MiniTensor(t.tensor([0.1, 0.5, 2.0]))
seed2 = resolve_end_grad(node, explicit)
print(f"Explicit seed: {seed2}")
assert t.allclose(seed2, explicit.array)

# Test path 2b: shape mismatch raises
bad = MiniTensor(t.tensor([1.0, 2.0]))
try:
    resolve_end_grad(node, bad)
    assert False, "Should have raised"
except AssertionError as e:
    print(f"Caught expected AssertionError: {e}")

print("resolve_end_grad works correctly on both paths.")